In [ ]:
# ── WIDGETS ──────────────────────────────────────────────────
dbutils.widgets.text("catalog_param", "my_assessment")
dbutils.widgets.text("schema_param", "silver")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────
from pyspark.sql.functions import col, current_timestamp, lit, max
from delta.tables import DeltaTable
from datetime import datetime

In [ ]:
# ── STEP 1: TRACK LAST RUN TIME ───────────────────────────────
# We store the last run time in a Delta table
# Every run reads this, processes only new/changed records
# Then updates it with current time

watermark_table = f"{catalog}.silver.pipeline_watermark"

# Create watermark table if it doesn't exist
if not spark.catalog.tableExists(watermark_table):
    spark.sql(f"""
        CREATE TABLE {watermark_table} (
            table_name STRING,
            last_run_time TIMESTAMP
        )
    """)
    # Insert epoch start for all 4 tables (first run = process everything)
    spark.sql(f"""
        INSERT INTO {watermark_table} VALUES
        ('customers',   '1900-01-01 00:00:00'),
        ('products',    '1900-01-01 00:00:00'),
        ('orders',      '1900-01-01 00:00:00'),
        ('order_items', '1900-01-01 00:00:00')
    """)
    print("✅ Watermark table created")
else:
    print("✅ Watermark table exists")

In [ ]:
# ── STEP 2: FUNCTION TO GET LAST RUN TIME ─────────────────────
def get_last_run_time(table_name):
    result = spark.sql(f"""
        SELECT last_run_time 
        FROM {watermark_table}
        WHERE table_name = '{table_name}'
    """).collect()
    return result[0][0]

In [ ]:
# ── STEP 3: FUNCTION TO UPDATE WATERMARK ──────────────────────
def update_watermark(table_name, run_time):
    spark.sql(f"""
        UPDATE {watermark_table}
        SET last_run_time = '{run_time}'
        WHERE table_name = '{table_name}'
    """)
    print(f"✅ Watermark updated for {table_name} → {run_time}")

In [ ]:
# ── STEP 4: INCREMENTAL MERGE FUNCTION ───────────────────────
def incremental_merge(table_name, primary_key):
    
    print(f"\n── Processing {table_name} ──")
    
    # 1. Get last run time
    last_run_time = get_last_run_time(table_name)
    current_run_time = datetime.now()
    print(f"   Last run time : {last_run_time}")
    print(f"   Current time  : {current_run_time}")
    
    # 2. Read from Bronze
    df_bronze = spark.table(f"{catalog}.bronze.{table_name}")
    
    # 3. Filter ONLY new or changed records
    # created_at > last_run_time → new records
    # updated_at > last_run_time → changed records
    df_incremental = df_bronze.filter(
        (col("created_at") > lit(last_run_time)) |
        (col("updated_at") > lit(last_run_time))
    )
    
    incremental_count = df_incremental.count()
    print(f"   New/changed records found: {incremental_count}")
    
    # 4. If no new records → skip
    if incremental_count == 0:
        print(f"   ⏭️ No new records — skipping {table_name}")
        return
    
    # 5. Clean the incremental data
    df_cleaned = df_incremental \
        .dropDuplicates([primary_key]) \
        .dropna(subset=[primary_key]) \
        .drop("ingestion_date", "source_path")
    
    # 6. Full table name
    full_table_name = f"{catalog}.silver.{table_name}_cleaned"
    
    # 7. MERGE into silver
    if spark.catalog.tableExists(full_table_name):
        # Table exists → UPSERT
        delta_table = DeltaTable.forName(spark, full_table_name)
        
        delta_table.alias("target").merge(
            df_cleaned.alias("source"),
            f"target.{primary_key} = source.{primary_key}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        
        print(f"   ✅ MERGED {incremental_count} records into {full_table_name}")
    
    else:
        # Table doesn't exist → full load
        df_cleaned.write.mode("overwrite") \
            .saveAsTable(full_table_name)
        print(f"   ✅ CREATED {full_table_name} with {incremental_count} records")
    
    # 8. Update watermark
    update_watermark(table_name, current_run_time)

In [ ]:
# ── STEP 5: RUN FOR ALL 4 TABLES ─────────────────────────────
incremental_merge("customers",    primary_key="customer_id")
incremental_merge("products",     primary_key="product_id")
incremental_merge("orders",       primary_key="order_id")
incremental_merge("order_items",  primary_key="order_item_id")

In [ ]:
# ── STEP 6: VERIFY ────────────────────────────────────────────
print("\n── Watermark Status ──")
spark.sql(f"SELECT * FROM {watermark_table}").display()

print("\n── Silver Tables ──")
spark.sql(f"SHOW TABLES IN {catalog}.silver").display()